# Exercise 13: Robot-Learning Environment

This notebook demonstrates the reference `RobotLearningEnv` implementation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from ch13_guide.envs.reaching_env import ReachingConfig, ReachingEnv
from ch13_guide.utils import proportional_controller, rollout

In [ ]:
config = ReachingConfig(
    max_steps=100,
    observation_noise_std=0.01,
    action_noise_std=0.002,
    action_delay_steps=0,
)
env = ReachingEnv(config=config, render_mode="rgb_array")

obs, info = env.reset(
    seed=3,
    options={
        "initial_position": np.array([-0.7, -0.5], dtype=np.float32),
        "goal": np.array([0.7, 0.6], dtype=np.float32),
    },
)
print(obs)
print(info)

In [ ]:
trajectory = [env.position.copy()]
records = []
while True:
    action = proportional_controller(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    trajectory.append(env.position.copy())
    records.append({"reward": reward, **info})
    if terminated or truncated:
        break

trajectory = np.asarray(trajectory)
print("Success:", info["is_success"])
print("Steps:", info["step_count"])
print("Safety interventions:", sum(r["safety_filter_activated"] for r in records))
print("Action saturations:", sum(r["action_saturated"] for r in records))

In [ ]:
frame = env.render()
plt.figure(figsize=(5, 5))
plt.imshow(frame)
plt.axis("off")
plt.title("Final environment state")
plt.show()

In [ ]:
results = rollout(
    ReachingEnv(config),
    proportional_controller,
    episodes=50,
    seed=10,
)
success_rate = np.mean([r["success"] for r in results])
mean_steps = np.mean([r["steps"] for r in results])
print(f"Success rate: {success_rate:.3f}")
print(f"Mean episode length: {mean_steps:.1f}")

## Instructor check

Students should verify that:

- success produces `terminated=True`;
- the time limit produces `truncated=True`;
- `raw_action` and `executed_action` can differ;
- reward components sum to the scalar reward;
- safety-filter activations are logged.